In [10]:
# Calculating feature importance of the dataset
#  Only look at the dataset with valid genres for the time being
import pandas as pd 
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import DictVectorizer


In [ ]:
data = pd.read_csv('data/songs_with_super_genres.csv')

valid_genre_data_mask = pd.isna(data['super_genre']) == False
valid_genre_data = data[valid_genre_data_mask]
valid_genre_data

# Split the data into training set 80% and test set 20%
df_train_full, df_test = train_test_split(valid_genre_data, test_size=0.2, random_state=1)
# Further split the training set into training set 75% and validation set 25%
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=1)

# train_test_split() also shuffles the data randomly before splitting
# so resetting the indexes as follows:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

#  extract the target labels (the genre column) as Numpy arrays
y_train = df_train['super_genre'].values
y_val = df_val['super_genre'].values
y_test = df_test['super_genre'].values

#  drop the target label from the data
#  for now dont include one-hot enocodings for nationality and detected language, maybe come back and include one hot encodings?
del df_train['super_genre']
del df_val['super_genre']
del df_test['super_genre']

cols = valid_genre_data.select_dtypes(include=np.number).columns.tolist()
print(cols)
#  ['Rank', 'Danceability', 'Energy', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Valence', 
# 'Points (Total)', 'num_languages', 'track_popularity', 'artist_followers', 'artist_popularity', 'Loudness']

['Rank', 'Danceability', 'Energy', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Valence', 'Points (Total)', 'num_languages', 'track_popularity', 'artist_followers', 'artist_popularity', 'Loudness']


In [11]:
# change the unique genres into numerical labels by numbering the super genres from 0 to n-1, n being the total number of unique super genres
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

#  gets the numeric feature columns and converts the resulting dataframe into a list of dictionaries, e.g., 'danceability': 0.5, 'energy': 0.8, ...
train_dicts = df_train[cols].to_dict(orient='records')

#  DictVectorizer will convert the list of dictionaries into a numeric vector similar to one-hot encoding for categorical variables, a dense Numpy array
#  fit_transform both learns the structure of waht features exists and converts the training data
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

# we do a similar thing for the validation data
val_dict = df_val[cols].to_dict(orient='records')
X_val = dv.transform(val_dict)  # only transform, don't fit again


In [14]:
# Feature importance using Random Forest Classifier
rf = RandomForestClassifier(random_state=1)
rf.fit(X_train, y_train)
importances = rf.feature_importances_
print(importances)

df_importances = pd.DataFrame()
df_importances['feature'] = dv.feature_names_
df_importances['importance'] = importances
df_importances.sort_values(by='importance', ascending=False, inplace=True)
print(df_importances)

[0.07855884 0.08262364 0.07887034 0.01605744 0.08523837 0.04812991
 0.04801992 0.10427404 0.07966028 0.17487903 0.13474247 0.00898191
 0.05996381]
              feature  importance
9    artist_followers    0.174879
10  artist_popularity    0.134742
7         Speechiness    0.104274
4            Loudness    0.085238
1        Danceability    0.082624
8             Valence    0.079660
2              Energy    0.078870
0        Acousticness    0.078559
12   track_popularity    0.059964
5      Points (Total)    0.048130
6                Rank    0.048020
3    Instrumentalness    0.016057
11      num_languages    0.008982
